## Core Mechanics & Theory

In CPython, variables do not store values directly. They hold references to objects in memory.


## 1. The PyObject Structure

Every Python object is represented internally by a C struct such as `PyObject` or `PyVarObject`.

- `ob_refcnt`: reference counter used by the garbage collector.
- `ob_type`: pointer to the object's type information.


```text
Variable 'a' (pointer) ---> [PyObject Header | Value Payload]
```

## 2. Identity vs. Equality

- `id(obj)`: returns the memory address of the object.
- `is`: checks whether two names point to the same object.
- `==`: checks whether the values are equal.


In [4]:
a=1
print(id(a))
type(a)

2007148396784


int

In [12]:
import sys

In [13]:
def diagnose(obj1, obj2):
    print(f"memory of the obj1: {hex(id(obj1))} \n and memory of the object 2 {hex(id(obj2))}")
    print(f"do they share same memory?{obj1 is obj2}")
    print(f"do they match the values{obj1==obj2}")
    print(f"refference count of obj1 :{sys.getrefcount(obj1)} \n and for the obj2 {sys.getrefcount(obj2)}")

In [14]:
diagnose(129,128)

memory of the obj1: 0x1d3535e10f0 
 and memory of the object 2 0x1d3535e10d0
do they share same memory?False
do they match the valuesFalse
refference count of obj1 :69 
 and for the obj2 157


In [15]:
diagnose(129,129)

memory of the obj1: 0x1d3535e10f0 
 and memory of the object 2 0x1d3535e10f0
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :72 
 and for the obj2 72


#### Note

Objects created during runtime do not always reuse the same cached identity as small integers.


In [ ]:
int("128") is int("128") #only from -5 to 256

True

In [23]:
diagnose(500,500)
int("500") is int("500")

memory of the obj1: 0x1d359864490 
 and memory of the object 2 0x1d359864490
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :7 
 and for the obj2 7


False

In [16]:
diagnose("dad","dad")

memory of the obj1: 0x1d359903cb0 
 and memory of the object 2 0x1d359903cb0
do they share same memory?True
do they match the valuesTrue
refference count of obj1 :7 
 and for the obj2 7


In [18]:
def dynamic(x):
    return x*1

x=0
while dynamic(x) is x:
    x-=1
print("lower boundary is ",x+1)

x=0
while dynamic(x) is x:
    x+=1
print("upper boundary is ",x-1)

lower boundary is  -5
upper boundary is  256


`sys.intern()` tells Python to reuse one shared string object for identical strings.


In [21]:
s1= "".join(["hello","world","!"])
s2= "".join(["hello","world","!"])

print(s1==s2)
print(s1 is s2)

s1_in= sys.intern(s1)
s2_in= sys.intern(s2)
print(s1_in is s2_in)

True
False
True


## Mutability, References & Copies

Understanding mutability at the memory level is important when working with custom data structures and tensor memory layouts.


## 1. Mutability vs. Immutability Under the Hood

- Immutable types such as `int`, `float`, `str`, `tuple`, and `frozenset` cannot change in place.
- Mutable types such as `list`, `dict`, and `set` can be modified in place.


## 2. The Reference Trap: Reassignment vs. In-Place Mutation

- Reassignment creates a new object and updates the variable reference.
- In-place mutation updates the existing object, so other references see the change.


## 3. Shallow Copy vs. Deep Copy

- Shallow copy creates a new outer container but keeps references to the same inner objects.
- Deep copy recursively duplicates all nested objects.


In [ ]:
def test_mutation():
    l1= [1,2,3]
    l2=l1
    l1+=[4]
    
    print(l1 is l2)#mutable
    
    tup1=(1,2,3)
    tup2=tup1
    
    tup1+=(4,)
    print(tup1 is tup2)#immutable creates new obj
test_mutation()

True
False


In [28]:
matrix = [[0]*3]*3
matrix

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [29]:
matrix[0][0]=1
matrix

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]

In [30]:
for i in range(3):
    print(id(matrix[i][0]),"/n")

2007148396784 /n
2007148396784 /n
2007148396784 /n


In [31]:
for i in range(3):
    print(id(matrix[0][i]),"/n")

2007148396784 /n
2007148396752 /n
2007148396752 /n


In [50]:
def deep_copy(a,memo=None):
    if memo is None:
        memo={}
    if isinstance(a,(int,float,str,bool)) or a is None:
        return a
    
    old_id=id(a)
    if old_id in memo:
        return memo[old_id]
    
    #list
    if isinstance(a,list):
        new_obj=[]
        memo[old_id]=new_obj
        
        for item in a:
            new_obj.append(deep_copy(item,memo))
        return new_obj
    #dict
    if isinstance(a,dict):
        new_obj={}
        memo[old_id]=new_obj
        for key, val in a.items():
            new_key=deep_copy(key)
            new_val = deep_copy(val)
            new_obj[new_key]=new_val
        return new_obj 
    if isinstance(a, set):
        new_obj = set()
        memo[old_id] = new_obj

        for item in a:
            new_obj.add(deep_copy(item, memo))

        return new_obj

    # Tuple
    if isinstance(a, tuple):
        new_obj = tuple(deep_copy(item, memo) for item in a)
        memo[old_id] = new_obj

        return new_obj

    raise TypeError(f"Unsupported type: {type(a)}")
    

In [51]:
a = [1, 2]
a.append(a)  # Circular reference
nested = {"key": [1, (2, 3), {4, 5}], "self": a}
cloned = deep_copy(nested)

In [53]:
cloned is not nested

True

In [54]:
cloned["key"][0] == nested["key"][0]


True

In [55]:
cloned["key"][1] is nested["key"][1] 


False

In [56]:

cloned["self"][2] is cloned["self"] 

True

## Truth Value Testing

When Python evaluates an expression like `if x:`, it checks:

1. `x.__bool__()` if it exists.
2. Otherwise, `x.__len__()`.
3. If neither is defined, the object is considered truthy by default.

Common falsy values include `None`, `False`, `0`, `0.0`, `""`, `()`, `[]`, `{}`, and `set()`.


In [61]:
for i in (None, False, 0, 0.0, 0j,"", (), [], {}, set()):
    if i:
        print("it will not print this")

## Short-Circuit Evaluation

Python logical operators do not always return `True` or `False`. They return the operand that determined the result.

- `x and y`: returns `x` if `x` is falsy; otherwise returns `y`.
- `x or y`: returns `x` if `x` is truthy; otherwise returns `y`.


## 3. The Walrus Operator (`:=`)

The walrus operator assigns a value and returns it at the same time.

- It is useful for avoiding repeated computation in loops and conditions.
- The assigned variable can be reused later in the surrounding scope.


In [62]:
name = input("Enter your name: ")

while name != "quit":
    print("Hello", name)
    name = input("Enter your name: ")

Hello hi
Hello quite


In [65]:
while (name:=input("enter your name"))!='quit':
    print("name is",name)

name is ji


## Custom Object Truthiness Engine

Create a class that makes truthiness depend on both the status code and the payload.

- The object is truthy only when `status_code == 200` and the payload has at least one item.
- Otherwise, it evaluates as falsy.


In [69]:
class DataPacket:
    def __init__(self,payload, status_code):
        self.payload=payload
        self.status_code=status_code

    def __len__(self):
        if self.payload is None:
            return 0
        return len(self.payload)
    
    def __bool__(self):
        return self.status_code == 200 and len(self) > 0

In [70]:
p1 = DataPacket([1, 2, 3], 200)
p2 = DataPacket([], 200)
p3 = DataPacket([1, 2, 3], 404)
p4 = DataPacket(None, 200)

print(bool(p1))  # True
print(bool(p2))  # False
print(bool(p3))  # False
print(bool(p4))  # False

True
False
False
False


## Exercise 2: Short-Circuit Pipeline Evaluator

Write a single-line assignment that selects:

1. `override_port` when it is a positive integer.
2. Otherwise `default_port` when it is a positive integer.
3. Otherwise `8080`.


## condition and value or fallback

In [80]:
def constraint(config: dict=None):
    port=config["override_port"]>=0 and config["override_port"] or config["default_port"]>=0 and config["default_port"] or 8080
    print(port)

In [81]:
constraint({"override_port":-1,"default_port":-1})

8080


In [82]:
constraint({"override_port":0,"default_port":90})

90


## Exercise 3: Stream Tokenizer with Walrus (`:=`)

Write a parser that extracts all numbers written in the form `[#123]` from a string.

- Use a loop with an assignment expression.
- Return the extracted integers as a list.


In [1]:
import re

def parse_tokens(raw_string: str):
    pattern = re.compile(r"\[#(\d+)\]")
    numbers = []
    pos = 0

    while (match := pattern.search(raw_string, pos)):
        numbers.append(int(match.group(1)))
        pos = match.end()

    return numbers

In [5]:
#alternatively
import re

def parse_tokens_findall(raw_string: str):
    return [int(x) for x in re.findall(r"\[#(\d+)\]", raw_string)]

In [2]:
raw = "Log 01: [#42] succeeded. Log 02: [#108] failed. Log 03: [#999] pending."

print(parse_tokens(raw))

[42, 108, 999]


In [6]:
raw = "Log 01: [#42] succeeded. Log 02: [#108] failed. Log 03: [#999] pending."

print(parse_tokens_findall(raw))

[42, 108, 999]
